# Evaluation and Convergence Plots for LLaDA-Inspired BERT Diffusion

This notebook is focused on analyzing iterative denoising behavior in Colab. It runs generation, collects convergence logs, compares ablation settings, and visualizes trends such as confidence growth and masked-token decay.

## 1. Install dependencies and clone the repo

In [ ]:
!pip -q install -U pip
!pip -q install -U transformers datasets torch tqdm matplotlib pandas seaborn

from pathlib import Path
import sys

REPO_URL = "https://github.com/lekkalapudiswetha-work/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM.git"
REPO_DIR = Path("/content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM")

if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd /content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM
!pip -q install -e .

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

## 2. Imports and config

In [ ]:
from dataclasses import dataclass
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from llada_bert import AblationRunner, BertMaskedLMWrapper, DiffusionNoiseScheduler, ExperimentConfig, IterativeDenoisingSampler

sns.set_theme(style="whitegrid")


@dataclass
class EvalConfig:
    model_name: str = "bert-base-uncased"
    checkpoint_path: str | None = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    prompt: str = "language models become more robust when"
    sequence_length: int = 24
    steps: int = 12
    threshold: float = 0.85
    temperature: float = 0.9
    top_k: int = 25
    num_samples: int = 3
    remask_strategy: str = "low_confidence"


cfg = EvalConfig()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

resolved_model_name = cfg.checkpoint_path or cfg.model_name
print("model:", resolved_model_name)
print("device:", cfg.device)
cfg

## 3. Run one generation pass and collect convergence logs

In [ ]:
wrapper = BertMaskedLMWrapper(model_name=resolved_model_name, device=cfg.device)
scheduler = DiffusionNoiseScheduler(total_steps=cfg.steps, base_threshold=cfg.threshold)
sampler = IterativeDenoisingSampler(
    model=wrapper,
    scheduler=scheduler,
    threshold=cfg.threshold,
    top_k=cfg.top_k,
)

result = sampler.sample(
    prompt=cfg.prompt,
    batch_size=cfg.num_samples,
    sequence_length=cfg.sequence_length,
    steps=cfg.steps,
    temperature=cfg.temperature,
    remask_strategy=cfg.remask_strategy,
)

for i, text in enumerate(result.texts, start=1):
    print(f"sample {i}: {text}")

log_df = pd.DataFrame(result.logger.as_rows())
log_df

## 4. Plot confidence and token dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.lineplot(data=log_df, x="step", y="mean_confidence", marker="o", ax=axes[0])
axes[0].set_title("Mean Confidence by Step")

sns.lineplot(data=log_df, x="step", y="masked_tokens", marker="o", ax=axes[1])
axes[1].set_title("Masked Tokens Remaining")

sns.barplot(data=log_df, x="step", y="changed_tokens", ax=axes[2])
axes[2].set_title("Changed Tokens per Step")

plt.tight_layout()
plt.show()

## 5. Compare thresholds and step counts with ablations

In [ ]:
runner_config = ExperimentConfig(
    model_name=resolved_model_name,
    device=cfg.device,
    steps=cfg.steps,
    sequence_length=cfg.sequence_length,
    temperature=cfg.temperature,
    threshold=cfg.threshold,
    top_k=cfg.top_k,
    seed=cfg.seed,
    num_samples=1,
    remask_strategy=cfg.remask_strategy,
)

runner = AblationRunner(runner_config)
ablation_grid = {
    "threshold": [0.75, 0.85, 0.92],
    "steps": [8, 12, 16],
}

ablation_results = runner.run_ablation_grid(cfg.prompt, ablation_grid)
len(ablation_results)

## 6. Build a summary table

In [ ]:
summary_rows = []
for item in ablation_results:
    summary_rows.append({
        "threshold": item["config"]["threshold"],
        "steps": item["config"]["steps"],
        "final_mean_confidence": item["summary"].get("final_mean_confidence"),
        "final_masked_tokens": item["summary"].get("final_masked_tokens"),
        "final_changed_tokens": item["summary"].get("final_changed_tokens"),
        "sample_text": item["texts"][0],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.sort_values(["threshold", "steps"]).reset_index(drop=True)

## 7. Plot ablation heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

confidence_pivot = summary_df.pivot(index="threshold", columns="steps", values="final_mean_confidence")
mask_pivot = summary_df.pivot(index="threshold", columns="steps", values="final_masked_tokens")

sns.heatmap(confidence_pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Final Mean Confidence")

sns.heatmap(mask_pivot, annot=True, fmt=".1f", cmap="YlOrRd_r", ax=axes[1])
axes[1].set_title("Final Masked Tokens")

plt.tight_layout()
plt.show()

## 8. Inspect the best settings

In [ ]:
best_confidence = summary_df.sort_values("final_mean_confidence", ascending=False).head(5)
best_confidence

## 9. Optional: compare multiple prompts

In [ ]:
prompts = [
    "language models become more useful when",
    "reasoning improves if a model can",
    "iterative refinement helps text generation by",
]

prompt_rows = []
for prompt in prompts:
    run = runner.run_single(prompt)
    prompt_rows.append({
        "prompt": prompt,
        "sample": run["texts"][0],
        "final_mean_confidence": run["summary"].get("final_mean_confidence"),
        "final_masked_tokens": run["summary"].get("final_masked_tokens"),
    })

pd.DataFrame(prompt_rows)

## 10. Suggested next analysis

- compare base BERT vs your fine-tuned checkpoint
- track prompt-specific stability across random seeds
- add lexical diversity metrics for generated samples
- export plots to Drive for experiment reports